# Practice workflow (mock of the hackathon template)
Run cells top to bottom. Set `LLM_PROVIDER=mock` in the first cell to test without spending tokens.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.path.abspath('main.py')))
sys.path.insert(0, '.')
# os.environ['LLM_PROVIDER'] = 'mock'   # uncomment: no LLM calls
import pandas as pd
pd.set_option('display.width', 200)

## 1. Explore the data (point-in-time)

In [ ]:
from src.utils import data, pit, universe
# Raw official format (long, codes) vs our normalised view (the adapter in src/utils/universe.py)
print(data.read_assets(end_date="2025-10-31").tail(3))
meta = universe.assets_meta(); meta

In [ ]:
as_of = '2025-10-31'
lv = pit.assets_asof(as_of).rename(columns=universe.names())
(lv.iloc[-252:] / lv.iloc[-252]).plot(figsize=(12, 5), legend=False, title=f'12m to {as_of}');

In [ ]:
from src.agents.tools import market_indicators_table, macro_snapshot_table, recent_news_text
print(market_indicators_table(as_of))
print(macro_snapshot_table(as_of))
print(recent_news_text(as_of, 7))

In [ ]:
from src.utils.news import find_topics, read_news
print(find_topics("central bank", "inflation"))
read_news("2025-10-20", "2025-10-31", topics=["Central Bank Interest Rates / Policy"],
          title_contains="Fed").to_pandas()[["published_at", "version", "version_role", "title"]]

### Polymarket: day-one check
`src/utils/polymarket.py` was written from the PDF without access to the library. Run this once at the hackathon: if the column names or attributes differ, fix `_real_expectations()` (nothing else uses the library).

In [ ]:
from src.utils import polymarket as pm
print("mode:", pm.mode())                       # 'real' when ecb_polymarket_api is installed
if pm.mode() == "real":
    from ecb_polymarket_api import discover_events, get_prices_by_event
    T = pm.cutoff("2026-03-31")
    evs = discover_events(tags=["european-central-bank"], start_date_max=T.date(), end_date_min=T.date())
    for e in evs[:5]:
        print(e.id, e.title, e.schedule.end_date, "neg_risk:", e.trading.neg_risk)
    ev = evs[0]
    print([(m.id, m.group_item_title) for m in ev.markets])
    df = get_prices_by_event(ev, end=T)
    print(df.columns.tolist(), df.index.max())  # are the columns market ids, titles or questions?
from src.agents.tools import policy_expectations_text
print(policy_expectations_text("2026-03-31"))

## 2. Quant core: risk of the strategic benchmark

In [ ]:
from src.config import LOOKBACK_DAYS
from src.utils.analytics import portfolio_metrics, check_constraints
b = universe.strategic_benchmark()            # rule-based: config.BENCHMARK_BUCKETS
rets = pit.returns_base_ccy_asof(as_of, LOOKBACK_DAYS)
print(check_constraints(b, meta))
portfolio_metrics(b, rets, meta)

## 3. Run the agents

In [ ]:
from src.agents.graph import build_workflow
from src.core.session import start_session
from src.core.run import run_date
from src.core.results import write_weights
from src.config import MODEL_ID, REPORTING_DATES
workflow = build_workflow()
workflow.get_graph().draw_mermaid()  # paste into mermaid.live to see the graph

In [ ]:
session = start_session()
result = run_date(workflow, as_of, MODEL_ID, session)
print(result['analyst_summary'][:2000])

In [ ]:
pd.DataFrame({'name': universe.names(), 'weight': result['approved_weights'],
              'benchmark': universe.strategic_benchmark()}).sort_values('weight', ascending=False)

In [ ]:
print(result['risk_commentary'])
result['adjustments']

In [ ]:
# The submission: all 6 dates in ONE session (as in the official multi_agents_market.ipynb)
results = {d: run_date(workflow, d, MODEL_ID, session) for d in REPORTING_DATES}
write_weights(results, session)          # weights.csv + prints the Langfuse session URL

# If a date fails, rerun only that date in the SAME session, then write again:
# results["2025-11-28"] = run_date(workflow, "2025-11-28", MODEL_ID, session)
# write_weights(results, session)